In [ ]:
%%bash
git fetch origin main

### Скорости

В файле `galaxies.csv` лежит список галактик из каталога Siena Galaxy Atlas 2020 года. Каждая галактика в этом списке характеризуется именем и двумя экваториальными координатами - прямое восхождение и склонение.

<details>
<summary>Откуда данные?</summary>

Статья: https://ui.adsabs.harvard.edu/abs/2023ApJS..269....3M

Сайт обзора: https://www.legacysurvey.org/sga/sga2020/

Скачать данные можно тут (но они уже лежат у вас на компьютере): https://vizier.unistra.fr/viz-bin/VizieR?-source=J/ApJS/269/3&-to=3

</details>

Из другого обзора у нас есть следующая галактика:
- Прямое восхождение: $3.0785451\degree$
- Склонение: $31.061563\degree$
- Картинка: 

![](images/galaxy.png)

Нужно определить имя этой галактики. Как будем действовать?

<details>
<summary>Спойлер (если ничего не придумывается)</summary>

1. Выбрать порог, после которого считаем галактики достаточно близкими.
2. Посчитать расстояние между этой галактикой и каждой галактикой из списка.
3. Посмотреть глазами на все галактики, находящиеся внутри порога.
4. Взять имя той, которая больше всего похожа на нужную.

</details>

### <отступление про расстояния не сфере>

На плоскости, если известны координаты двух точек $x_1, y_1$ и $x_2$, $y_2$, расстояние между ними вычисляется из теоремы Пифагора:

$$
d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}
$$

На сфере это, строго говоря, не так:

![](images/3d_sphere_distance.png)

<details>
<summary style="color: gray;">Это, кстати, Desmos!</summary>

Позволяет рисовать произвольно сложные 3D графики: https://www.desmos.com/3d/xxphp4awjf

</details>

Расстояние на сфере между двумя точками (расстояние большого круга) считается по формуле 

$$
\theta = arccos(\sin(\delta_1)\sin(\delta_2) + \cos(\delta_1)\cos(\delta_2)\cos(\alpha_1 - \alpha_2))
$$

### </отступление про расстояния на сфере>

Попробуем найти галактики из SGA, соответствующие нашей галактике.

<div class="alert alert-block alert-warning" style="margin-top: 20px">

<font size=4>**Задание 1**</font>     

В файле `data/galaxies.csv` лежит CSV файл, в котором есть 3 столбца:
- `name` - имя галактики
- `ra` - прямое восхождение в градусах
- `dec` - склонение в градусах

Найти и вывести имена всех галактик, находящихся на расстоянии менее 10 угловых минут от галактики с координатами выше.

*Отдельно можно засечь на таймере, сколько времени займёт исполнение кода.*

</div>

In [ ]:
ra_g = 3.078541
dec_g = 31.061563
radius = 10/60

with open("data/sga.csv") as file:
    for line in file.readlines():
        name, ra_str, dec_str = line.split(",")
        ra = float(ra_str)
        dec = float(dec_str)

        if (ra - ra_g) ** 2 + (dec - dec_g) ** 2 < radius ** 2:
            print(name, ra, dec)

### Можно ли быстрее?

В нашем примере порядок числа галактик - 100 тысяч. В современных обзорах число объектов измеряется миллионами, сотнями миллионов и миллиардами:
- SDSS-V - ~6 млн галактик
- [DESI](https://www.desi.lbl.gov/the-desi-survey/) - ~30 млн объектов
- [Gaia DR3](https://www.cosmos.esa.int/web/gaia/dr3) - ~1.46 млрд объектов

На таких масштабах и с более сложными операциями время обработки увеличивается на порядки.

Все слышали, что Python на самом деле медленный. Давайте сменим язык программирования на C и посмотрим, что будет.

<details>
<summary>Если вдруг написание кода на C не сработает</summary>

```c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>

#define MAX_LINE_LENGTH 256
#define MAX_NAME_LENGTH 100
#define MAX_GALAXIES 1000000
#define SEARCH_RADIUS_DEG 0.166666

// Структура для хранения информации о галактике
typedef struct {
    char name[MAX_NAME_LENGTH];
    double ra;      // прямое восхождение в градусах
    double dec;     // склонение в градусах
} Galaxy;

// Функция для вычисления простого евклидова расстояния
// Все координаты в градусах
double euclidean_distance(double ra1, double dec1, double ra2, double dec2) {
    double delta_ra = ra2 - ra1;
    double delta_dec = dec2 - dec1;
    
    // Простое расстояние по теореме Пифагора
    return sqrt(delta_ra * delta_ra + delta_dec * delta_dec);
}

// Функция для чтения CSV файла
int read_csv(const char *filename, Galaxy galaxies[], int max_galaxies) {
    FILE *file = fopen(filename, "r");
    if (file == NULL) {
        printf("Ошибка: не удалось открыть файл %s\n", filename);
        return -1;
    }
    
    char line[MAX_LINE_LENGTH];
    int count = 0;
    
    // Если в файле есть заголовок, раскомментируйте следующую строку
    // fgets(line, MAX_LINE_LENGTH, file);
    
    while (fgets(line, MAX_LINE_LENGTH, file) != NULL && count < max_galaxies) {
        // Удаляем символ новой строки
        line[strcspn(line, "\n")] = 0;
        
        // Пропускаем пустые строки
        if (strlen(line) == 0) continue;
        
        // Парсим строку CSV
        char *token = strtok(line, ",");
        if (token == NULL) continue;
        
        // Копируем имя
        strncpy(galaxies[count].name, token, MAX_NAME_LENGTH - 1);
        galaxies[count].name[MAX_NAME_LENGTH - 1] = '\0';
        
        // Читаем прямое восхождение
        token = strtok(NULL, ",");
        if (token == NULL) continue;
        galaxies[count].ra = atof(token);
        
        // Читаем склонение
        token = strtok(NULL, ",");
        if (token == NULL) continue;
        galaxies[count].dec = atof(token);
        
        count++;
    }
    
    fclose(file);
    return count;
}

// Функция для парсинга координат из строки (формат: "HH:MM:SS" или "DD:MM:SS")
double parse_coordinate(const char *coord_str, int is_ra) {
    double degrees = 0.0;
    double hours = 0.0, minutes = 0.0, seconds = 0.0;
    
    // Пытаемся распарсить формат HH:MM:SS или DD:MM:SS
    if (sscanf(coord_str, "%lf:%lf:%lf", &hours, &minutes, &seconds) == 3) {
        if (is_ra) {
            // Для прямого восхождения: часы -> градусы (1 час = 15 градусов)
            degrees = (hours + minutes / 60.0 + seconds / 3600.0) * 15.0;
        } else {
            // Для склонения: градусы, минуты, секунды
            degrees = fabs(hours) + minutes / 60.0 + seconds / 3600.0;
            if (hours < 0) degrees = -degrees;
        }
    } else {
        // Если не удалось распарсить как HH:MM:SS, пробуем как десятичное число
        degrees = atof(coord_str);
    }
    
    return degrees;
}

int main(int argc, char *argv[]) {
    char filename[256];
    char target_ra_str[50], target_dec_str[50];
    double target_ra, target_dec;
    Galaxy *galaxies;
    int galaxy_count;

    if (argc != 4) {
        printf("Использование: %s <файл.csv> <RA> <DEC>\n", argv[0]);
        printf("Пример: %s galaxies.csv \"12:30:45\" \"+45:30:00\"\n", argv[0]);
        printf("RA и DEC можно указывать в формате:\n");
        printf("  - десятичные градусы: 12.5125 45.5\n");
        printf("  - часовой/градусный формат: 12:30:45 +45:30:00\n");
        return 1;
    }
    
    strcpy(filename, argv[1]);
    strcpy(target_ra_str, argv[2]);
    strcpy(target_dec_str, argv[3]);
    
    // Парсим координаты целевой галактики
    target_ra = parse_coordinate(target_ra_str, 1);
    target_dec = parse_coordinate(target_dec_str, 0);
    
    printf("Целевая галактика: RA = %.6f°, DEC = %.6f°\n", target_ra, target_dec);
    printf("Поиск галактик в пределах %.1f градуса...\n\n", SEARCH_RADIUS_DEG);

    galaxies = malloc((size_t)MAX_GALAXIES * sizeof(Galaxy));
    if (galaxies == NULL) {
        fprintf(stderr, "Ошибка: не удалось выделить память под каталог\n");
        return 1;
    }

    galaxy_count = read_csv(filename, galaxies, MAX_GALAXIES);

    if (galaxy_count <= 0) {
        printf("Не удалось прочитать галактики из файла или файл пуст.\n");
        free(galaxies);
        return 1;
    }
    
    printf("Всего загружено галактик: %d\n", galaxy_count);
    printf("\nНайденные галактики в пределах %.1f градуса:\n", SEARCH_RADIUS_DEG);
    printf("================================================\n\n");
    
    int found = 0;
    
    // Проходим по всем галактикам и проверяем расстояние
    for (int i = 0; i < galaxy_count; i++) {
        double distance = euclidean_distance(target_ra, target_dec, 
                                            galaxies[i].ra, galaxies[i].dec);
        
        if (distance <= SEARCH_RADIUS_DEG) {
            printf("Имя: %s\n", galaxies[i].name);
            printf("  RA: %.6f° (%.6f°), DEC: %.6f°\n", 
                   galaxies[i].ra, galaxies[i].ra, galaxies[i].dec);
            printf("  Расстояние: %.4f градусов\n", distance);
            printf("  ΔRA: %.4f°, ΔDEC: %.4f°\n", 
                   fabs(galaxies[i].ra - target_ra),
                   fabs(galaxies[i].dec - target_dec));
            printf("----------------------------------------\n");
            found++;
        }
    }
    
    if (found == 0) {
        printf("Галактики в пределах %.1f градуса не найдены.\n", SEARCH_RADIUS_DEG);
    } else {
        printf("\nВсего найдено галактик: %d\n", found);
    }

    free(galaxies);
    return 0;
}
```

Компиляция:
```bash
gcc -o sga sga.c -lm
```

Запуск:
```bash
./sga data/sga.csv 3.078541 31.061563
```

</details>


### <отступление на то, как рисовать красивые изображения неба>

Для отображения изображений неба есть проект Aladin. У него несколько частей:
- Сайт для отображения карты неба (Aladin Lite): https://aladin.cds.unistra.fr/AladinLite/
- Aladin Desktop - приложение на компьютер для просмотра карт неба: https://aladin.cds.unistra.fr/java/nph-aladin.pl?frame=downloading
- Виджет для Python-ноутбуков `ipyaladin`: https://aladin.cds.unistra.fr/AladinLite/ipyaladin/

Последний нам интересен больше всего - он позволяет простым образом нарисовать нам карту неба. Представим, что мы хотим посмотреть на Крабовидную туманность:

In [ ]:
from astropy.coordinates import Angle
from ipyaladin import Aladin

aladin = Aladin(
    target="Crab",
    fov=Angle(10/60, "deg"),
)
aladin

Нужно всего два модуля:
- `astropy` - модуль с астрономическими утилитами. Подробнее мы про него поговорим на следующих школах.
- `ipyaladin` - модуль, занимающийся непосредственно рисованием.

Если хочется вместо имени объекта использовать конкретные координаты, это можно сделать вот так:

In [ ]:
from astropy.coordinates import Angle, SkyCoord
from ipyaladin import Aladin

aladin = Aladin(
    target=SkyCoord(283.396, 33.029, unit="deg"),
    fov=Angle(10/60, "deg"),
)
aladin

Этот виджет позволяет делать много чего - покликайте на разные кнопки.

### </отступление на то, как рисовать красивые изображения неба>

Результаты получили, давайте посмотрим на сами галактики, чтобы понять, какая из них правильная:

In [ ]:
aladin = Aladin(
    target=SkyCoord(2.9988702, 30.9973874, unit="deg"),
    fov=Angle(10/60, "deg"),
)
aladin

In [ ]:
aladin = Aladin(
    target=SkyCoord(2.9887549, 30.9985883, unit="deg"),
    fov=Angle(10/60, "deg"),
)
aladin

In [ ]:
aladin = Aladin(
    target=SkyCoord(3.0785801, 31.0610539, unit="deg"),
    fov=Angle(10/60, "deg"),
)
aladin